# V2 Pipeline - Phase 1: Phân tích Khám phá Dữ liệu (EDA) & Kỹ thuật Đặc trưng (Combined (MIMIC + PTB-XL))

Thực hiện theo chỉ đạo chuyên môn:
1. **Nạp dữ liệu Combined (MIMIC + PTB-XL)**: `data/features/combined_mimic_ptbxl_advanced.csv`.
2. **Rà soát thuộc tính phân loại**: One-Hot Encoding cho các biến phân loại (nếu có).
3. **Khảo sát độ biến thiên**: Thống kê độ lệch chuẩn & độ biến thiên của các đặc trưng thô.
4. **Mean Imputation**: Xử lý nhiễu bằng `SimpleImputer(strategy='mean')`.
5. **Tạo 2 tập Scaled V2**: Min-Max Scaling (`v2_combined_minmax_scaled.csv`) vs Z-Score (`v2_combined_zscore_scaled.csv`).

In [ ]:
import os, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
warnings.filterwarnings('ignore')
print('✅ Nạp thành công thư viện EDA V2 cho Combined (MIMIC + PTB-XL)!')


### 1. Nạp Dữ Liệu Thô & Kiểm tra Biến Phân Loại

In [ ]:
data_candidates = ['../../../data/features/combined_mimic_ptbxl_advanced.csv', '../../data/features/combined_mimic_ptbxl_advanced.csv', 'data/features/combined_mimic_ptbxl_advanced.csv']
data_path = next((p for p in data_candidates if os.path.exists(p)), None)
df_raw = pd.read_csv(data_path)
print(f'Nạp dữ liệu thành công từ: {data_path}')
print(f'Kích thước dữ liệu thô: {df_raw.shape[0]} mẫu | {df_raw.shape[1]} cột')
cat_cols = df_raw.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    df_processed = pd.get_dummies(df_raw, columns=cat_cols, drop_first=True)
    print('✅ Đã áp dụng One-Hot Encoding!')
else:
    df_processed = df_raw.copy()
    print('ℹ️ Tất cả đặc trưng đều là dạng số (Numerical).')


### 2. Thống Kê Độ Biến Thiên Dữ Liệu Thô (Variance Analysis)

In [ ]:
feature_cols = [c for c in df_processed.columns if c != 'status']
X_raw = df_processed[feature_cols]
y = df_processed['status']
stats_df = pd.DataFrame({
    'Mean': X_raw.mean(),
    'Std_Dev': X_raw.std(),
    'Variance': X_raw.var(),
    'Min': X_raw.min(),
    'Max': X_raw.max(),
    'Range': X_raw.max() - X_raw.min()
}).sort_values(by='Variance', ascending=False)
print('--- BẢNG THỐNG KÊ ĐỘ BIẾN THIÊN CÁC ĐẶC TRƯNG THÔ COMBINED (MIMIC + PTB-XL) ---\n')
print(stats_df.round(4).to_string())


### 3. Xử lý Mean Imputation & Xuất Dữ Liệu Scaled (Min-Max vs Z-Score)

In [ ]:
# === BƯỚC LỌC NHIỄU OUTLIER THEO HƯỚNG DẪN CỦA ANH NGUYỄN VĂN VƯỢNG ===
# 1. Lọc bỏ các mẫu có giá trị Min-Max Scaling > 0.95 (Chấm ngoại lệ ở đỉnh)
# 2. Lọc bỏ các mẫu có giá trị Z-Score Normalization > 4.7 (Dị biệt biến thiên cực đoan)
mask_minmax = (df_minmax.drop(columns=['status']) <= 0.95).all(axis=1)
mask_zscore = (df_zscore.drop(columns=['status']) <= 4.7).all(axis=1)
valid_mask = mask_minmax & mask_zscore

n_orig = len(df_raw)
n_clean = valid_mask.sum()
print(f'Mẫu dữ liệu ban đầu: {n_orig}')
print(f'Số dòng nhiễu Outlier bị loại bỏ: {n_orig - n_clean} ({(n_orig - n_clean)/n_orig*100:.2f}%)')
print(f'Số mẫu sạch còn lại: {n_clean}')

df_minmax = df_minmax[valid_mask].reset_index(drop=True)
df_zscore = df_zscore[valid_mask].reset_index(drop=True)
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X_raw)
X_imputed_df = pd.DataFrame(X_imputed, columns=feature_cols)

minmax_scaler = MinMaxScaler()
df_minmax = pd.DataFrame(minmax_scaler.fit_transform(X_imputed_df), columns=feature_cols)
df_minmax['status'] = y.values

zscore_scaler = StandardScaler()
df_zscore = pd.DataFrame(zscore_scaler.fit_transform(X_imputed_df), columns=feature_cols)
df_zscore['status'] = y.values

save_dir_candidates = ['../../../data/features', '../../data/features', 'data/features']
save_dir = next((d for d in save_dir_candidates if os.path.exists(os.path.dirname(d))), '../../data/features')
os.makedirs(save_dir, exist_ok=True)

path_minmax = os.path.join(save_dir, 'v2_combined_minmax_scaled.csv')
path_zscore = os.path.join(save_dir, 'v2_combined_zscore_scaled.csv')
df_minmax.to_csv(path_minmax, index=False)
df_zscore.to_csv(path_zscore, index=False)
print(f'✅ Đã lưu Min-Max Scaled vào: {path_minmax}')
print(f'✅ Đã lưu Z-Score Scaled vào: {path_zscore}')


### 4. Trực quan hóa So sánh Phân phối Thô vs Min-Max vs Z-Score

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
top_5_var_cols = stats_df.index[:5].tolist()
sns.boxplot(data=X_raw[top_5_var_cols], ax=axes[0], palette='Set2')
axes[0].set_title('Top 5 Đặc trưng thô Combined (MIMIC + PTB-XL) (Variance lớn nhất)')
axes[0].tick_params(axis='x', rotation=30)
sns.boxplot(data=df_minmax[top_5_var_cols], ax=axes[1], palette='Set2')
axes[1].set_title('Sau khi Min-Max Scaling [0, 1]')
axes[1].tick_params(axis='x', rotation=30)
sns.boxplot(data=df_zscore[top_5_var_cols], ax=axes[2], palette='Set2')
axes[2].set_title('Sau khi Z-Score Normalization')
axes[2].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()
